# 03 — Đánh giá hệ thống (TV5)

Notebook thử nghiệm phần **đánh giá**: từ khoảng cách Hamming của từng cặp ảnh tính
**TP / TN / FP / FN → Accuracy, Sensitivity, Specificity**, vẽ **đường cong ROC**, tính **AUC**
và chọn **ngưỡng (threshold)** tối ưu.

Quy ước: lớp dương = **Similar**; dự đoán `Similar` khi `normalized Hamming distance <= threshold`.

> Nếu module `WaveletHash` của TV3 đã hoàn thiện, notebook tự dùng hash của nhóm; nếu chưa,
> dùng bản tham chiếu của TV5 (db4, LL 8x8, ngưỡng median). Lý thuyết: `docs/03_research/evaluation_methods.md`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from experiments.evaluate_dataset import (
    DEFAULT_LABELS, DEFAULT_THRESHOLD, build_hash_fn, compute_pair_hashes,
    load_pairs, run_wavelet_sweep, variation_breakdown,
)
from src.evaluation import (
    compute_roc, evaluate_at_threshold, leave_one_out_accuracy,
    plot_confusion_matrix, plot_distance_distribution, plot_roc_curve,
    threshold_sweep,
)

pd.set_option("display.precision", 4)

## 1. Đọc dataset và tính Hamming distance cho từng cặp

In [2]:
pairs = load_pairs(DEFAULT_LABELS, DEFAULT_LABELS.parent)
hash_fn, source = build_hash_fn("auto", wavelet="db4", hash_size=8)
print("Nguồn hash:", source)

rows = compute_pair_hashes(pairs, hash_fn)
df = pd.DataFrame(rows)[["pair_id", "label", "variation", "hamming_distance", "normalized_distance"]]
df["label"] = df["label"].map({1: "similar", 0: "dissimilar"})
df

[Thông báo] Module WaveletHash của TV3 chưa sẵn sàng (ImportError) -> dùng bản tham chiếu builtin của TV5.
Nguồn hash: Bản tham chiếu TV5 (wavelet=db4, LL 8x8, 64 bit, ngưỡng median)


,pair_id,label,variation,hamming_distance,normalized_distance
0,similar/pair_01,similar,angle,22,0.3438
1,similar/pair_02,similar,brightness,0,0.0000
2,similar/pair_03,similar,size,1,0.0156
3,similar/pair_04,similar,noise,20,0.3125
4,similar/pair_05,similar,contrast,2,0.0312
5,similar/pair_06,similar,combo(angle+brightness+noise),24,0.3750
6,similar/pair_07,similar,angle,28,0.4375
7,similar/pair_08,similar,brightness,2,0.0312
8,similar/pair_09,similar,size,0,0.0000
9,similar/pair_10,similar,noise,18,0.2812


## 2. Phân bố khoảng cách của hai nhóm

In [3]:
y = np.array([r["label"] for r in rows])
d = np.array([r["normalized_distance"] for r in rows])

print(f"Similar    : trung bình = {d[y == 1].mean():.4f}, trung vị = {np.median(d[y == 1]):.4f}")
print(f"Dissimilar : trung bình = {d[y == 0].mean():.4f}, trung vị = {np.median(d[y == 0]):.4f}")
plot_distance_distribution(y, d)
plt.show()

Similar    : trung bình = 0.1828, trung vị = 0.1562
Dissimilar : trung bình = 0.3844, trung vị = 0.3906


## 3. Đường cong ROC và AUC

Mỗi ngưỡng `t` cho một điểm `(FPR, TPR)`. Ngưỡng tối ưu chọn theo **Youden's J = TPR − FPR**
(= Sensitivity + Specificity − 1).

In [4]:
roc = compute_roc(y, d)
print(f"AUC = {roc.auc:.4f}")
print(f"Ngưỡng Youden = {roc.youden_threshold:.4f}  (J = {roc.youden_j:.4f})")
plot_roc_curve(roc)
plt.show()

AUC = 0.8250
Ngưỡng Youden = 0.1562  (J = 0.5000)


## 4. Chỉ số tại ngưỡng mặc định (0.25 của TV4) và ngưỡng tối ưu

In [5]:
m_def = evaluate_at_threshold(y, d, DEFAULT_THRESHOLD)
m_opt = evaluate_at_threshold(y, d, roc.youden_threshold)

summary = pd.DataFrame([m_def.to_dict(), m_opt.to_dict()], index=["mặc định", "tối ưu (Youden)"])
summary[["threshold", "tp", "tn", "fp", "fn", "accuracy", "sensitivity", "specificity", "precision", "f1"]]

,threshold,tp,tn,fp,fn,accuracy,sensitivity,specificity,precision,f1
mặc định,0.2500,5,10,0,5,0.75,0.5,1.0,1.0,0.6667
tối ưu (Youden),0.1562,5,10,0,5,0.75,0.5,1.0,1.0,0.6667


In [6]:
plot_confusion_matrix(m_opt.tp, m_opt.tn, m_opt.fp, m_opt.fn,
                      title=f"Confusion matrix (ngưỡng {m_opt.threshold:.4f})")
plt.show()

## 5. Ước lượng công bằng: leave-one-out

Chọn ngưỡng và đánh giá trên **cùng** 20 cặp cho kết quả *lạc quan*. Leave-one-out chọn ngưỡng trên 19 cặp
còn lại rồi dự đoán cặp bị bỏ ra — gần với hiệu năng thật trên dữ liệu mới hơn.

In [7]:
loo = leave_one_out_accuracy(y, d)
print(f"Accuracy trên toàn bộ (ngưỡng tối ưu) : {m_opt.accuracy:.4f}")
print(f"Accuracy leave-one-out                : {loo:.4f}")

Accuracy trên toàn bộ (ngưỡng tối ưu) : 0.7500
Accuracy leave-one-out                : 0.6500


## 6. Quét ngưỡng: Sensitivity và Specificity đánh đổi nhau thế nào

In [8]:
thresholds = np.linspace(0, 0.6, 25)
sweep = threshold_sweep(y, d, thresholds)
plt.figure(figsize=(7, 4.5))
plt.plot(thresholds, [m.sensitivity for m in sweep], label="Sensitivity")
plt.plot(thresholds, [m.specificity for m in sweep], label="Specificity")
plt.plot(thresholds, [m.accuracy for m in sweep], "--", label="Accuracy")
plt.axvline(roc.youden_threshold, color="black", ls=":", label="Ngưỡng Youden")
plt.xlabel("Ngưỡng normalized Hamming distance")
plt.ylabel("Giá trị")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 7. Các cặp bị phân loại sai và ảnh hưởng của từng loại biến thể

In [9]:
df["dự đoán"] = np.where(d <= m_opt.threshold, "similar", "dissimilar")
df["đúng?"] = df["label"] == df["dự đoán"]
print("Các cặp SAI tại ngưỡng tối ưu:")
display(df[~df["đúng?"]])

pd.DataFrame(
    variation_breakdown(rows, m_opt.threshold),
    columns=["biến thể", "số cặp", "nhận đúng", "sensitivity"],
)

Các cặp SAI tại ngưỡng tối ưu:


,pair_id,label,variation,hamming_distance,normalized_distance,dự đoán,đúng?
0,similar/pair_01,similar,angle,22,0.3438,dissimilar,False
3,similar/pair_04,similar,noise,20,0.3125,dissimilar,False
5,similar/pair_06,similar,combo(angle+brightness+noise),24,0.3750,dissimilar,False
6,similar/pair_07,similar,angle,28,0.4375,dissimilar,False
9,similar/pair_10,similar,noise,18,0.2812,dissimilar,False


,biến thể,số cặp,nhận đúng,sensitivity
0,angle,2,0,0.0000
1,brightness,2,2,1.0000
2,combo,1,0,0.0000
3,contrast,1,1,1.0000
4,noise,2,0,0.0000
5,size,2,2,1.0000


## 8. So sánh hiệu quả phân loại của các loại Wavelet

In [10]:
sweep_w = pd.DataFrame(run_wavelet_sweep(pairs, hash_size=8))
sweep_w.sort_values("auc", ascending=False)

,wavelet,auc,threshold,accuracy,sensitivity,specificity,loo_accuracy
3,db8,0.840,0.1406,0.75,0.5,1.0,0.65
6,coif1,0.830,0.4297,0.80,0.9,0.7,0.65
2,db4,0.825,0.1562,0.75,0.5,1.0,0.65
4,sym2,0.815,0.2812,0.85,0.8,0.9,0.85
1,db2,0.815,0.2812,0.85,0.8,0.9,0.85
5,sym4,0.735,0.0781,0.75,0.5,1.0,0.70
0,haar,0.310,0.0156,0.50,0.0,1.0,0.40


## 9. Nhận xét

- Xem số liệu chính thức và phân tích trong `docs/06_results/` (tự cập nhật bằng
  `python experiments/evaluate_dataset.py --sweep-wavelets`).
- Dataset chỉ có **20 cặp**, nên mọi chỉ số có sai số lớn: 1 cặp sai thay đổi Accuracy 5%.
- Kết quả phụ thuộc mạnh vào cấu hình hash (loại wavelet, kích thước LL, cách lượng tử hóa) — xem mục 8.